# Step 02: Detect and Audit Dataset Anomalies
### Automated 8-Rule Quality Inspector, Multi-Panel Diagnostic Graphs & Visual Sample Audits
Directly integrated with `multi_class_train_rfdetr.ipynb`'s Step B verification (Cell 12).

### 8 Anomaly Detection Rules:
1. **Degenerate Boxes**: Width or height <= 0.
2. **Micro-Clicks**: Accidental clicks with width/height < 4px or area < 16px.
3. **Giant Boxes**: Oversized bounding boxes covering > 95% of image area.
4. **Severe Image Blur**: Blurry images detected via Laplacian variance threshold (< 80.0).
5. **Edge-Cropped / Truncated Boxes**: Bounding boxes abruptly cropped by image borders (touching x=0, y=0, x+w>=W, or y+h>=H).
6. **Exact Duplicates**: Overlapping boxes of the same class with IoU > 0.95.
7. **Loose Margin Padding**: Excessive background padding around tags.
8. **Dense Query Overlaps**: Overly clustered overlapping tags.

### Key Outputs:
- Multi-Panel Anomaly Diagnostic Graphs (Distribution, Aspect Ratios, Areas, Density).
- Visual Sample Inspector displaying flagged anomalies (red boxes & yellow reason tags) vs clean boxes (green).
- `anomalies_manifest.json`: Complete record of all flagged anomaly boxes.
- `cleaned_dataset/`: Sanitized COCO annotations strictly excluding all anomalies.

In [ ]:
# STEP 0 - Environment Dependencies
%pip install -q pycocotools Pillow matplotlib pandas numpy tabulate tqdm

In [ ]:
# CELL 1 - Imports, VRAM Monitor & Logger Initialization
import os, sys, json, time, math, copy, logging, random, shutil, warnings
warnings.filterwarnings("ignore", message=".*meshgrid.*")
warnings.filterwarnings("ignore", message=".*torch.cuda.amp.autocast.*")
warnings.filterwarnings("ignore", message=".*max_detection_threshold.*")
warnings.filterwarnings("ignore", category=FutureWarning)
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from collections import defaultdict, Counter
import urllib.parse
from urllib.parse import urlparse, unquote
from concurrent.futures import ThreadPoolExecutor, as_completed

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(line_buffering=True)
os.environ["PYTHONUNBUFFERED"] = "1"

# Configure multiprocessing sharing strategy to prevent 'Too many open files' error
import torch.multiprocessing as mp
try:
    mp.set_sharing_strategy('file_system')
except Exception:
    pass

try:
    import resource
    rlimit = resource.getrlimit(resource.RLIMIT_NOFILE)
    resource.setrlimit(resource.RLIMIT_NOFILE, (max(rlimit[0], 4096), max(rlimit[1], 4096)))
except Exception:
    pass

import cv2, torch, requests, supervision as sv
# Neutralize Matplotlib interactive callback to permanently prevent _draw_all_if_interactive crash
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.ioff()
plt.close('all')
try:
    _ip = get_ipython()
    if _ip and hasattr(_ip, 'events'):
        _ip.events.callbacks['post_execute'] = [
            _cb for _cb in _ip.events.callbacks.get('post_execute', [])
            if 'draw_all' not in getattr(_cb, '__name__', '')
        ]
except Exception:
    pass
import matplotlib.patches as patches, numpy as np, pandas as pd
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image, ImageDraw, ImageFont
from dotenv import load_dotenv
from tqdm.auto import tqdm

# Auto-flushing console and file logger
class FlushHandler(logging.StreamHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

class FlushFileHandler(logging.FileHandler):
    def emit(self, record):
        super().emit(record)
        self.flush()

logger = logging.getLogger("rfdetr_multi_class")
logger.setLevel(logging.INFO)
logger.handlers.clear()
logger.propagate = False
log_format = "%(asctime)s | %(levelname)-8s | %(message)s"
console_handler = FlushHandler(sys.stdout)
console_handler.setFormatter(logging.Formatter(log_format))
logger.addHandler(console_handler)

def print_vram_usage(tag="Status"):
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated() / (1024**3)
        total = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        logger.info(f"[VRAM - {tag}] {torch.cuda.get_device_name(0)}: {alloc:.2f} GB / {total:.2f} GB allocated")
    else:
        logger.info(f"[VRAM - {tag}] Running on CPU")

logger.info("=" * 65)
logger.info(f"[CELL 1] Core Libraries & Logger Ready (PyTorch: {torch.__version__}, CUDA: {torch.cuda.is_available()})")
print_vram_usage("Init")
logger.info("=" * 65)


import io
def show_plt_figure(fig=None):
    """Guarantees inline plot display in Jupyter under headless and Agg backends."""
    if fig is None:
        fig = plt.gcf()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=100)
    buf.seek(0)
    pil_img = Image.open(buf)
    plt.close(fig)
    display(pil_img)

In [ ]:
# CELL 2 - Paths, Output Setup & Field Configuration
REPO_ROOT = next((p.resolve() for p in [Path("."), Path(".."), Path("../..")] if (p / "coco_files").exists() or (p / ".git").exists()), Path(".").resolve())
INPUT_JSON_DIR = Path("coco_files") if Path("coco_files").exists() else (REPO_ROOT / "coco_files")

# Auto-discover active images pool directory across standard pipeline locations
candidate_image_dirs = [
    PIPELINE_DIR / "images",
    Path("/home/jupyter/multi_class_rfdetr/images"),
    REPO_ROOT / "multi_class_train_rfdetr" / "images",
    INPUT_JSON_DIR / "images",
    REPO_ROOT / "images",
    Path("images")
]
IMAGES_DIR = next((p for p in candidate_image_dirs if p.exists() and any(p.glob("*"))), candidate_image_dirs[0])
os.makedirs(IMAGES_DIR, exist_ok=True)

# COCO annotation field mappings
IMAGE_FIELD = "image_id"
CATEGORY_FIELD = "category_id"
BBOX_FIELD = "bbox"

# ==============================================================================
# MASTER ANOMALY AUDIT & SANITIZATION TOGGLES
# ==============================================================================
FILTER_EDGE_CROPPED_BOXES = False   # Set False: Border tags are legitimate valid objects (keep in training)
FILTER_MICRO_DEGENERATE = False     # Set False: Small/distant tags are legitimate valid objects (keep in training)
SHOW_ANOMALY_SAMPLES_PER_CATEGORY = 5   # Display exactly 5 sample images with tags for each anomaly category
MAX_DISPLAY_WIDTH = 750                 # Display width for inline notebook previews

PIPELINE_NAME = "multi_class_rfdetr"
PIPELINE_DIR = REPO_ROOT / PIPELINE_NAME
MODEL_VERSION = os.getenv("MODEL_VERSION", "v2")
VERSION_TAG = MODEL_VERSION if MODEL_VERSION else "v1"
VERSION_DIR = PIPELINE_DIR / VERSION_TAG
os.makedirs(VERSION_DIR, exist_ok=True)

# Unified Version Folder: all anomaly audits, manifests and cleaned datasets live here
OUTPUT_ANOMALIES_DIR = VERSION_DIR / "anomalies_audit"
CLEANED_DATASET_DIR = VERSION_DIR / "cleaned_dataset"
OUTPUT_ANOMALIES_DIR.mkdir(parents=True, exist_ok=True)
CLEANED_DATASET_DIR.mkdir(parents=True, exist_ok=True)

logger.info(f"Input JSON Dir:        {INPUT_JSON_DIR.resolve()}")
logger.info(f"Active Images Pool:    {IMAGES_DIR.resolve()} ({len(list(IMAGES_DIR.glob('*'))):,} cached images)")
logger.info(f"Anomalies Output Dir:  {OUTPUT_ANOMALIES_DIR.resolve()}")
logger.info(f"Cleaned Dataset Dir:   {CLEANED_DATASET_DIR.resolve()}")
logger.info(f"Show Samples Per Rule: {SHOW_ANOMALY_SAMPLES_PER_CATEGORY} sample images per anomaly category")


In [ ]:
# CELL 4 - Read all annotation JSON files from coco_files/
logger.info("=" * 70)
logger.info(f"[CELL 4] Searching for annotation JSON files in: {INPUT_JSON_DIR}")
logger.info("=" * 70)

if not INPUT_JSON_DIR.exists():
    INPUT_JSON_DIR.mkdir(parents=True, exist_ok=True)

json_files = sorted(list(INPUT_JSON_DIR.glob("*.json")))
logger.info(f"   - Found {len(json_files)} JSON file(s) in {INPUT_JSON_DIR}")

raw_records = []
raw_images = {}  # image_id / file_name -> {"id": ..., "width": ..., "height": ..., "file_name": ...}
raw_categories = {}  # category_id -> category_name

if len(json_files) <= 5:
    for jf in json_files:
        logger.info(f"   - Reading: {jf.name} ({jf.stat().st_size / 1024:.1f} KB)")
else:
    logger.info(f"   - Ingesting {len(json_files)} JSON file(s)... (sample: {', '.join(p.name for p in json_files[:3])})")

for jf in tqdm(json_files, desc="[CELL 4] Reading JSON files", unit="file"):
    with open(jf, "r", encoding="utf-8") as f:
        content = f.read().strip()
        if not content:
            continue
        try:
            parsed = json.loads(content)
            if isinstance(parsed, dict):
                # Preserve full image metadata (width, height, file_name, id)
                for im in parsed.get("images", []):
                    if isinstance(im, dict):
                        iid = im.get("id")
                        if iid is not None:
                            raw_images[iid] = im
                            raw_images[str(iid)] = im
                        fn = im.get("file_name")
                        if fn:
                            raw_images[fn] = im
                            raw_images[Path(fn).name] = im
                            
                if "annotations" in parsed and isinstance(parsed["annotations"], list):
                    for c in parsed.get("categories", []):
                        if isinstance(c, dict) and "id" in c and "name" in c:
                            raw_categories[c["id"]] = c["name"]
                            raw_categories[str(c["id"])] = c["name"]
                    raw_records.extend(parsed["annotations"])
                else:
                    raw_records.append(parsed)
            elif isinstance(parsed, list):
                raw_records.extend(parsed)
        except json.JSONDecodeError:
            f.seek(0)
            for line_idx, line in enumerate(f):
                line = line.strip()
                if line:
                    try:
                        parsed_line = json.loads(line)
                        if isinstance(parsed_line, list):
                            raw_records.extend(parsed_line)
                        else:
                            raw_records.append(parsed_line)
                    except Exception as err:
                        logger.warning(f"Error decoding line {line_idx+1} in {jf.name}: {err}")

logger.info(f"Total raw annotation records loaded: {len(raw_records):,} across {len(raw_images):,} registered image metadata entries.")


In [ ]:

def extract_clean_filename(raw_val: Any, default_id: Any = None) -> str:
    """Safely extracts clean image filename, stripping URL query tokens and Azure SAS signatures."""
    if not raw_val:
        return f"{default_id}.jpg" if default_id is not None else "image.jpg"
    s = str(raw_val).strip()
    import urllib.parse
    from urllib.parse import urlparse, unquote
    s = unquote(s)
    s = s.split("?")[0].split("#")[0]
    if "&sig=" in s:
        s = s.split("&sig=")[0]
    p_name = Path(urlparse(s).path).name or Path(s).name
    if Path(p_name).stem.endswith("=") or p_name.endswith("="):
        return f"{default_id}.jpg" if default_id is not None else "image.jpg"
    if not p_name.lower().endswith((".jpg", ".jpeg", ".png", ".webp")):
        p_name = f"{p_name}.jpg"
    return p_name

# CELL 5 - Parse Annotations & Auto-Discover Multiple Categories
logger.info("=" * 70)
logger.info("[CELL 5] Parsing annotations and discovering categories dynamically...")
logger.info("=" * 70)

# 1. Pre-scan for Coordinate System (Normalized [0..1] vs Absolute Pixels)
sample_bboxes = [
    item.get(BBOX_FIELD, []) for item in raw_records 
    if isinstance(item, dict) and isinstance(item.get(BBOX_FIELD), list) and len(item.get(BBOX_FIELD)) == 4
]

is_normalized_dataset = False
if sample_bboxes:
    max_sample_val = 0.0
    for b in sample_bboxes[:3000]:
        try:
            vals = [float(v) for v in b]
            if vals:
                max_sample_val = max(max_sample_val, max(vals))
        except (ValueError, TypeError):
            continue
            
    # If all coordinates in sample are <= 1.05, the dataset is in normalized [0, 1] format!
    if max_sample_val > 0.0 and max_sample_val <= 1.05:
        is_normalized_dataset = True
        logger.warning("=" * 70)
        logger.warning(f"⚠️ DETECTED NORMALIZED [0..1] BOUNDING BOXES! (sample max coordinate = {max_sample_val:.4f})")
        logger.warning("   Auto-scaling coordinates to absolute pixel space using image width/height.")
        logger.warning("=" * 70)

STANDARD_CLASSES = ["blue_aisle", "blue_bay", "location_tag"]

def resolve_target_category(item, raw_cat_map=None) -> str:
    c = item.get("category_name")
    if not c and "category_id" in item and raw_cat_map:
        cid = item["category_id"]
        c = raw_cat_map.get(cid, raw_cat_map.get(str(cid)))
    if not c:
        c = item.get("category") or item.get("category_id")
    if isinstance(c, list):
        c = c[0] if len(c) > 0 else ""
    c_str = str(c).strip().lower()
    if "aisle" in c_str or c_str in ["blue_aisle", "blue-aisle", "aisle"]:
        return "blue_aisle"
    elif "bay" in c_str or "blue_tag" in c_str or c_str in ["blue_bay", "bay"]:
        return "blue_bay"
    elif "location" in c_str or "white" in c_str or "loc" in c_str or c_str in ["location_tag", "white_tag", "tag", "default"]:
        return "location_tag"
    cid = item.get("category_id")
    if cid is not None:
        try:
            cid_int = int(cid)
            if cid_int in [0, 1, 2] and raw_cat_map and cid_int in raw_cat_map:
                mapped = str(raw_cat_map[cid_int]).lower()
                if "aisle" in mapped: return "blue_aisle"
                if "bay" in mapped or "blue_tag" in mapped: return "blue_bay"
                if "location" in mapped or "white" in mapped: return "location_tag"
            if cid_int == 0: return "blue_aisle"
            elif cid_int == 1: return "blue_bay"
            elif cid_int == 2: return "location_tag"
        except (ValueError, TypeError):
            pass
    return "location_tag"

image_annotations: Dict[str, List[Dict[str, Any]]] = {}
category_counts: Dict[str, int] = {}
skipped_invalid_boxes = 0

for item in tqdm(raw_records, desc="[CELL 5] Parsing annotations", unit="rec"):
    if not isinstance(item, dict):
        continue
    img_id = item.get(IMAGE_FIELD)
    if not img_id:
        continue
    
    cat_name = resolve_target_category(item, raw_categories)
    category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

    raw_bbox = item.get(BBOX_FIELD, [])
    if not (isinstance(raw_bbox, list) and len(raw_bbox) == 4):
        continue
    
    try:
        x, y, w, h = [float(v) for v in raw_bbox]
    except (ValueError, TypeError):
        skipped_invalid_boxes += 1
        continue

    # Resolve true image dimensions from registered COCO metadata
    im_info = raw_images.get(img_id, raw_images.get(str(img_id), {}))
    img_w = float(im_info.get("width") or 1920)
    img_h = float(im_info.get("height") or 1080)

    # Scale normalized coordinates to absolute pixels if dataset is normalized
    if is_normalized_dataset or (x <= 1.05 and y <= 1.05 and w <= 1.05 and h <= 1.05 and (w > 0 or h > 0)):
        x = x * img_w
        y = y * img_h
        w = w * img_w
        h = h * img_h

    # Mathematically degenerate check (width or height must be positive)
    if w <= 0.0 or h <= 0.0:
        skipped_invalid_boxes += 1
        continue


    true_fn = extract_clean_filename(im_info.get("file_name") if im_info else (item.get("image_url") or item.get("coco_url") or f"{img_id}.jpg"), default_id=img_id)
    orig_url = (im_info.get("coco_url") if im_info else None) or item.get("image_url") or img_id
    ann_dict = {
        "id": item.get("id"),
        "image_id": img_id,
        "bbox": [round(x, 2), round(y, 2), round(w, 2), round(h, 2)],
        "category_name": cat_name,
        "area": float(item.get("area", w * h)),
        "segmentation": item.get("segmentation", []),
        "img_w": img_w,
        "img_h": img_h,
        "file_name": true_fn,
        "image_url": orig_url
    }
    
    if img_id not in image_annotations:
        image_annotations[img_id] = []
    image_annotations[img_id].append(ann_dict)

# Guarantee all 3 multi-class target categories
discovered_cats = set(category_counts.keys()).union(set(STANDARD_CLASSES))
for c in raw_categories.values():
    discovered_cats.add(resolve_target_category({"category_name": c}))
AUTO_CATEGORIES = sorted(list(discovered_cats))
if set(AUTO_CATEGORIES) == set(STANDARD_CLASSES):
    AUTO_CATEGORIES = list(STANDARD_CLASSES)
cat_to_id = {cat: idx for idx, cat in enumerate(AUTO_CATEGORIES)}
id_to_cat = {idx: cat for cat, idx in cat_to_id.items()}

for img_id, anns in image_annotations.items():
    for ann in anns:
        ann["category_id"] = cat_to_id[ann["category_name"]]
        ann["image_id"] = img_id

NUM_CLASSES = len(AUTO_CATEGORIES)
total_boxes = sum(len(v) for v in image_annotations.values())

logger.info(f"   - Unique images found:         {len(image_annotations):,}")
logger.info(f"   - Total valid bounding boxes:  {total_boxes:,}")
if skipped_invalid_boxes > 0:
    logger.info(f"   - Skipped non-positive boxes:  {skipped_invalid_boxes:,} (w<=0 or h<=0)")
logger.info(f"   - Discovered Categories ({NUM_CLASSES}):")
for cat, count in category_counts.items():
    logger.info(f"       - ID {cat_to_id[cat]}: '{cat}' ({count:,} boxes, {count/max(1, total_boxes)*100:.1f}%)")
logger.info(f"Discovered {NUM_CLASSES} categories: {AUTO_CATEGORIES}. Total boxes: {total_boxes:,}")
# Construct definitions for downstream sanitized dataset exports
categories_list = [{"id": idx, "name": cat, "supercategory": "none"} for cat, idx in cat_to_id.items()]
annotations_by_image = image_annotations
annotations_by_cat: Dict[str, List[Dict[str, Any]]] = defaultdict(list)
for anns in image_annotations.values():
    for a in anns:
        annotations_by_cat[a["category_name"]].append(a)


In [ ]:
# CELL 5 - Comprehensive Anomaly Detector Engine with Multi-Class Format Verification
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm
from collections import Counter, defaultdict
import os, sys
from pathlib import Path
from typing import Dict, List, Any, Optional, Tuple
from PIL import Image
import numpy as np

def compute_box_iou(boxA, boxB):
    xA = max(boxA[0], boxB[0])
    yA = max(boxA[1], boxB[1])
    xB = min(boxA[0] + boxA[2], boxB[0] + boxB[2])
    yB = min(boxA[1] + boxA[3], boxB[1] + boxB[3])
    interW = max(0.0, xB - xA)
    interH = max(0.0, yB - yA)
    interArea = interW * interH
    areaA = boxA[2] * boxA[3]
    areaB = boxB[2] * boxB[3]
    denom = areaA + areaB - interArea
    return interArea / denom if denom > 0 else 0.0

# Image path resolver for fast file lookups
search_dirs = [
    str(VERSION_DIR / "images") if "VERSION_DIR" in globals() else None,
    str(PIPELINE_DIR / "images") if "PIPELINE_DIR" in globals() else None,
    str(IMAGES_DIR) if "IMAGES_DIR" in globals() else "images",
    str(REPO_ROOT / "multi_class_train_rfdetr" / "images") if "REPO_ROOT" in globals() else None,
    str(INPUT_JSON_DIR / "images") if "INPUT_JSON_DIR" in globals() else None,
    str(Path("coco_files/images")),
    str(Path("images")),
    "/home/jupyter/multi_class_train_rfdetr/images",
    "/home/jupyter/coco_files/images",
    "/home/jupyter/images"
]
search_dirs = [d for d in search_dirs if d and os.path.exists(d)]

def resolve_image_path(img_identifier):
    img_str = str(img_identifier).split("?")[0].split("#")[0]
    import urllib.parse
    unquoted = urllib.parse.unquote(img_str)
    fname = Path(img_str).name
    unquoted_fname = Path(unquoted).name
    stem = Path(img_str).stem
    candidates = [unquoted_fname, fname, f"{stem}.jpg", f"{img_str}.jpg", img_str]
    for d in search_dirs:
        for c in candidates:
            p = os.path.join(d, c)
            if os.path.exists(p) and os.path.getsize(p) > 0:
                return p
    return None

def compute_single_image_blur(img_identifier) -> Tuple[str, float]:
    """Computes Laplacian variance sharpness score (lower = more blurry)."""
    local_p = resolve_image_path(img_identifier)
    if not local_p:
        return img_identifier, 999.0
    try:
        try:
            import cv2
            gray = cv2.imread(local_p, cv2.IMREAD_GRAYSCALE)
            if gray is not None:
                if gray.shape[1] > 640:
                    gray = cv2.resize(gray, (640, int(gray.shape[0] * 640 / gray.shape[1])), interpolation=cv2.INTER_AREA)
                return img_identifier, float(cv2.Laplacian(gray, cv2.CV_64F).var())
        except Exception:
            pass

        im = Image.open(local_p).convert("L")
        if im.width > 640:
            scale = 640 / im.width
            im = im.resize((640, int(im.height * scale)), Image.BILINEAR)
        arr = np.array(im, dtype=np.float64)
        lap = (
            -4.0 * arr[1:-1, 1:-1]
            + arr[:-2, 1:-1]
            + arr[2:, 1:-1]
            + arr[1:-1, :-2]
            + arr[1:-1, 2:]
        )
        return img_identifier, float(lap.var())
    except Exception:
        return img_identifier, 999.0

# 1. Fast indexed lookup of raw image dimensions
raw_rec_lookup = {}
for r in raw_records:
    if isinstance(r, dict):
        cu = r.get("coco_url")
        u = r.get("url")
        if cu: raw_rec_lookup[cu] = r
        if u: raw_rec_lookup[u] = r

# 2. Precompute blur scores across images concurrently with multi-threading and tqdm
image_source = annotations_by_image if "annotations_by_image" in globals() else image_annotations
unique_images = list(image_source.keys())

_image_blur_scores: Dict[str, float] = {}
BLUR_THRESHOLD = 80.0

logger.info(f"[CELL 5] Parallel blur analysis across {len(unique_images)} images using ThreadPoolExecutor...")
with ThreadPoolExecutor(max_workers=min(16, (os.cpu_count() or 4) * 2)) as executor:
    futures = {executor.submit(compute_single_image_blur, iid): iid for iid in unique_images}
    for fut in tqdm(as_completed(futures), total=len(unique_images), desc="[CELL 5] Blur & Sharpness Scan", unit="img"):
        try:
            iid, score = fut.result()
            _image_blur_scores[iid] = score
        except Exception:
            pass

flagged_anomalies = []
clean_annotations = []
total_boxes = 0

# 3. Comprehensive audit loop auditing ALL anomaly categories & multi-class format patterns
logger.info(f"[CELL 5] Executing comprehensive anomaly audit across {len(image_source)} image sets for all classes...")
for img_url, anns in tqdm(image_source.items(), desc="[CELL 5] Anomaly Detection Audit", unit="img"):
    im_meta = raw_images.get(img_url, raw_images.get(str(img_url), {})) if "raw_images" in globals() else {}
    img_w = float(im_meta.get("width") or (anns[0].get("img_w") if anns else 1920) or 1920)
    img_h = float(im_meta.get("height") or (anns[0].get("img_h") if anns else 1080) or 1080)
    
    blur_score = _image_blur_scores.get(img_url, 999.0)
    is_image_blurry = blur_score < BLUR_THRESHOLD
    num_anns = len(anns)
    
    for i, a in enumerate(anns):
        total_boxes += 1
        bbox = a.get("bbox", [])
        if len(bbox) != 4:
            flagged_anomalies.append({
                "annotation": a,
                "image_url": img_url,
                "reasons": ["Malformed bbox dimensions"],
                "primary_rule": "Malformed bbox dimensions",
                "bbox": [0,0,0,0],
                "category_name": a.get("category_name", "Unknown")
            })
            continue
            
        x, y, w, h = float(bbox[0]), float(bbox[1]), float(bbox[2]), float(bbox[3])
        area = w * h
        is_anomaly = False
        reasons = []
        cat_name = str(a.get("category_name", "")).lower().strip()
        
        # Rule 1: Degenerate Box (Zero or Negative Dimensions)
        if w <= 0 or h <= 0:
            is_anomaly = True
            reasons.append(f"Degenerate box (w={w:.1f}, h={h:.1f})")
            
        # Rule 2: Micro-Click / Sub-pixel Tag
        if (w < 2.0 or h < 2.0 or area < 4.0) and (w > 0 and h > 0):
            is_anomaly = True
            reasons.append(f"Micro-click (<2px, w={w:.1f}, h={h:.1f}, area={area:.1f})")
            
        # Rule 3: Giant Box Covering >95% of Canvas
        if w > 0.95 * img_w and h > 0.95 * img_h:
            is_anomaly = True
            reasons.append(f"Giant box covering >95% image (w={w:.1f}, h={h:.1f})")
            
        # Rule 4: Severe Image Blur
        if is_image_blurry:
            is_anomaly = True
            reasons.append(f"Blurry image (Laplacian var={blur_score:.1f} < {BLUR_THRESHOLD})")
            
        # Rule 5: Edge-Cropped / Border-Truncated Box
        is_cropped_left = x <= 1.0
        is_cropped_top = y <= 1.0
        is_cropped_right = (x + w) >= (img_w - 1.0)
        is_cropped_bottom = (y + h) >= (img_h - 1.0)
        if is_cropped_left or is_cropped_top or is_cropped_right or is_cropped_bottom:
            border_sides = []
            if is_cropped_left: border_sides.append("left")
            if is_cropped_top: border_sides.append("top")
            if is_cropped_right: border_sides.append("right")
            if is_cropped_bottom: border_sides.append("bottom")
            is_anomaly = True
            reasons.append(f"Cropped box at image border ({', '.join(border_sides)})")
            
        # Rule 6: Duplicate Box with High IoU
        if num_anns > 1:
            for j in range(num_anns):
                if i != j and a.get("category_name") == anns[j].get("category_name"):
                    b_box = anns[j].get("bbox", [])
                    if len(b_box) == 4 and abs(x - b_box[0]) < 15 and abs(y - b_box[1]) < 15:
                        iou = compute_box_iou(bbox, b_box)
                        if iou > 0.95:
                            is_anomaly = True
                            reasons.append(f"Duplicate box with IoU {iou:.2f}")
                            break
                            
            # Rule 7: Dense Cluster Overlap
            overlapping_count = 0
            for j in range(num_anns):
                if i != j:
                    b_box = anns[j].get("bbox", [])
                    if len(b_box) == 4:
                        if compute_box_iou(bbox, b_box) > 0.70:
                            overlapping_count += 1
                            if overlapping_count >= 3:
                                break
            if overlapping_count >= 3:
                is_anomaly = True
                reasons.append(f"Dense cluster overlap ({overlapping_count} overlapping boxes)")
                
        # Rule 8: Loose Shelf Padding Across Multiple Bays
        if w > 0.65 * img_w and h < 0.08 * img_h:
            is_anomaly = True
            reasons.append("Loose shelf padding across multiple bays")
            
        # Rule 9: Multi-Class Format & Geometry Pattern Checks (Across ALL Classes)
        aspect_ratio_h_w = h / max(w, 0.1)
        aspect_ratio_w_h = w / max(h, 0.1)

        # 9a. Out-of-Bounds Coordinates Check (All classes)
        if x < -2.0 or y < -2.0 or (x + w) > (img_w + 10.0) or (y + h) > (img_h + 10.0):
            is_anomaly = True
            reasons.append(f"[{cat_name}] Out-of-Bounds Coordinates (x={x:.0f}, y={y:.0f}, w={w:.0f}, h={h:.0f}, canvas={img_w:.0f}x{img_h:.0f})")

        # 9b. Format Pattern & Aspect Ratio Checks per Class
        if "location" in cat_name or cat_name in ["location_tag", "tag", "white_tag"]:
            if aspect_ratio_h_w > 4.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Vertical Aspect Ratio (h/w={aspect_ratio_h_w:.1f} > 4.0)")
            elif aspect_ratio_w_h > 8.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Horizontal Sliver (w/h={aspect_ratio_w_h:.1f} > 8.0)")
            if area > 0.35 * img_w * img_h:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Unusually Giant Area ({area/(img_w*img_h)*100:.1f}% of image)")

        elif "bay" in cat_name or cat_name in ["blue_bay", "bay"]:
            if aspect_ratio_h_w > 6.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Vertical Aspect Ratio (h/w={aspect_ratio_h_w:.1f} > 6.0)")
            elif aspect_ratio_w_h > 8.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Horizontal Sliver (w/h={aspect_ratio_w_h:.1f} > 8.0)")
            if area > 0.65 * img_w * img_h:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Unusually Giant Area ({area/(img_w*img_h)*100:.1f}% of image)")

        elif "aisle" in cat_name or cat_name in ["blue_aisle", "aisle"]:
            if aspect_ratio_h_w > 6.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Vertical Aspect Ratio (h/w={aspect_ratio_h_w:.1f} > 6.0)")
            elif aspect_ratio_w_h > 8.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Horizontal Sliver (w/h={aspect_ratio_w_h:.1f} > 8.0)")
            if area > 0.50 * img_w * img_h:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Unusually Giant Area ({area/(img_w*img_h)*100:.1f}% of image)")

        else:
            if aspect_ratio_h_w > 6.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Vertical Aspect Ratio (h/w={aspect_ratio_h_w:.1f} > 6.0)")
            elif aspect_ratio_w_h > 8.0:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Extreme Horizontal Sliver (w/h={aspect_ratio_w_h:.1f} > 8.0)")
            if area > 0.60 * img_w * img_h:
                is_anomaly = True
                reasons.append(f"[{cat_name}] Unusually Giant Area ({area/(img_w*img_h)*100:.1f}% of image)")
            
        if is_anomaly:
            primary = reasons[0].split("(")[0].strip() if reasons else "Other anomaly"
            flagged_anomalies.append({
                "annotation": a,
                "image_url": img_url,
                "reasons": reasons,
                "primary_rule": primary,
                "bbox": bbox,
                "category_name": a.get("category_name", "Unknown")
            })
        else:
            clean_annotations.append(a)

# 4. Compute Counts of Each Anomaly Category (Overall & Per Class)
rule_counts = Counter()
class_rule_counts = defaultdict(Counter)
class_box_counts = Counter()
class_anomaly_counts = Counter()

for anom in flagged_anomalies:
    c_name = str(anom.get("category_name", "unknown")).lower()
    class_anomaly_counts[c_name] += 1
    reasons = anom.get("reasons", ["Unspecified anomaly"])
    for r in reasons:
        r_name = r.split("(")[0].strip()
        rule_counts[r_name] += 1
        class_rule_counts[c_name][r_name] += 1

for anns in image_source.values():
    for a in anns:
        c_name = str(a.get("category_name", "unknown")).lower()
        class_box_counts[c_name] += 1

print()
print("=" * 85)
print(f"ANOMALY AUDIT COMPLETE: {len(flagged_anomalies):,} / {total_boxes:,} boxes flagged ({len(flagged_anomalies)/max(1, total_boxes)*100:.2f}%)")
print("=" * 85)
print(f"{'Anomaly Category / Rule':<50} | {'Count':<8} | Share (%)")
print("-" * 80)
for rule_name, count in rule_counts.most_common():
    share_pct = (count / max(1, total_boxes)) * 100
    print(f"  {rule_name:<48} | {count:<8,} | {share_pct:5.2f}%")
print("-" * 80)
print(f"  {'Total Clean Annotations Retained':<48} | {len(clean_annotations):<8,} | {len(clean_annotations)/max(1, total_boxes)*100:5.2f}%")
print("=" * 85)
print()

# 5. Class-by-Class Anomaly Breakdown Table
all_known_classes = sorted(list(set(class_box_counts.keys()).union(set(class_anomaly_counts.keys()))))
print("=" * 85)
print("CLASS-BY-CLASS ANOMALY BREAKDOWN TABLE")
print("=" * 85)
print(f"{'Class Name':<18} | {'Total Boxes':<12} | {'Anomalies':<10} | {'Clean Boxes':<12} | Anomaly Rate")
print("-" * 85)
for c in all_known_classes:
    t_cnt = class_box_counts[c]
    a_cnt = class_anomaly_counts[c]
    c_cnt = t_cnt - a_cnt
    a_rate = (a_cnt / max(1, t_cnt)) * 100
    print(f"  {c:<16} | {t_cnt:<12,} | {a_cnt:<10,} | {c_cnt:<12,} | {a_rate:6.2f}%")
    top_rules = [f"{r} ({cnt})" for r, cnt in class_rule_counts[c].most_common(3)]
    if top_rules:
        print(f"    └─ Top Anomaly Rules: {', '.join(top_rules)}")
print("=" * 85)
print()

# 6. Multi-Class Format Pattern Verification Report (Across ALL Classes)
def _pct(arr, q):
    if not arr: return 0.0
    s = sorted(arr)
    return s[min(int(len(s) * q), len(s) - 1)]

print("=" * 85)
print("MULTI-CLASS FORMAT PATTERN VERIFICATION REPORT (ALL CLASSES)")
print("=" * 85)
for c in all_known_classes:
    c_boxes = [a for anns in image_source.values() for a in anns if str(a.get("category_name", "")).lower() == c]
    if not c_boxes:
        continue
    c_widths = [float(a["bbox"][2]) for a in c_boxes if len(a.get("bbox", [])) == 4 and a["bbox"][2] > 0]
    c_heights = [float(a["bbox"][3]) for a in c_boxes if len(a.get("bbox", [])) == 4 and a["bbox"][3] > 0]
    c_ratios = [w / max(h, 0.1) for w, h in zip(c_widths, c_heights)]
    c_areas_pct = [float(a.get("area", a["bbox"][2] * a["bbox"][3])) / max(1.0, (float(a.get("img_w", 1920)) * float(a.get("img_h", 1080)))) * 100 for a in c_boxes if len(a.get("bbox", [])) == 4]
    
    c_anom_cnt = class_anomaly_counts[c]
    c_anom_rate = (c_anom_cnt / max(1, len(c_boxes))) * 100

    print(f"Class: [{c.upper()}] (Total: {len(c_boxes):,} boxes | Flagged Anomalies: {c_anom_cnt:,} [{c_anom_rate:.1f}%])")
    print(f"  • Width (px):        Min={min(c_widths):.1f}, Median={_pct(c_widths, 0.5):.1f}, p95={_pct(c_widths, 0.95):.1f}, Max={max(c_widths):.1f}")
    print(f"  • Height (px):       Min={min(c_heights):.1f}, Median={_pct(c_heights, 0.5):.1f}, p95={_pct(c_heights, 0.95):.1f}, Max={max(c_heights):.1f}")
    print(f"  • Aspect Ratio (w/h): Min={min(c_ratios):.2f}, Median={_pct(c_ratios, 0.5):.2f}, p95={_pct(c_ratios, 0.95):.2f}, Max={max(c_ratios):.2f}")
    print(f"  • Area (% canvas):   Min={min(c_areas_pct):.3f}%, Median={_pct(c_areas_pct, 0.5):.2f}%, p95={_pct(c_areas_pct, 0.95):.2f}%, Max={max(c_areas_pct):.1f}%")
    print("-" * 85)
print("=" * 85)
print()


In [ ]:
# CELL 6 - Comprehensive Anomaly Diagnostic Graphs
import matplotlib.pyplot as plt
from collections import Counter

def normalize_anomaly_category(reason: str) -> str:
    r = str(reason).split("(")[0].strip()
    if "Duplicate box" in r:
        return "Duplicate Box Overlap"
    if "Micro-click" in r:
        return "Micro-Click (<2px)"
    if "Degenerate box" in r:
        return "Degenerate Box (Zero Dim)"
    if "Cropped box at image border" in r or "border" in r.lower():
        return "Edge-Cropped / Border-Truncated Box"
    if "Blurry" in r:
        return "Severe Image Blur"
    if "Giant box" in r:
        return "Giant Box (>95% Image)"
    if "Dense cluster" in r:
        return "Dense Cluster Overlap"
    if "Loose shelf padding" in r:
        return "Loose Shelf Padding"
    if "Extreme Vertical Aspect" in r or "Extreme Vertical" in r:
        return r
    if "Extreme Horizontal Sliver" in r or "Horizontal Sliver" in r:
        return r
    if "Out-of-Bounds" in r:
        return r
    if "Unusually Giant Area" in r:
        return r
    return r

rule_counts = Counter()
for item in flagged_anomalies:
    for r in item["reasons"]:
        rule_counts[normalize_anomaly_category(r)] += 1

fig, axs = plt.subplots(2, 2, figsize=(18, 12))

# Panel 1: Anomaly Rule Frequency Breakdown
labels = list(rule_counts.keys())
counts = list(rule_counts.values())
if labels:
    sorted_idx = sorted(range(len(counts)), key=lambda k: counts[k], reverse=True)
    labels = [labels[i] for i in sorted_idx][:10]
    counts = [counts[i] for i in sorted_idx][:10]
    bars = axs[0, 0].barh(range(len(labels)), counts, color="#e74c3c", edgecolor="black", alpha=0.85)
    axs[0, 0].set_yticks(range(len(labels)))
    axs[0, 0].set_yticklabels(labels, fontsize=9)
    axs[0, 0].invert_yaxis()
    axs[0, 0].set_title(f"Top 10 Flagged Anomaly Rules (Total Anomaly Boxes: {len(flagged_anomalies):,})", fontsize=11, fontweight="bold")
    axs[0, 0].set_xlabel("Count of Flagged Boxes")
    for bar in bars:
        axs[0, 0].text(bar.get_width() + max(1, max(counts)*0.01), bar.get_y() + bar.get_height()/2, f"{int(bar.get_width()):,}", va="center", fontsize=8)
else:
    axs[0, 0].text(0.5, 0.5, "No Anomalies Detected", ha="center", va="center")

# Panel 2: Anomaly Rate (%) by Category / Class
if all_known_classes:
    cls_rates = [(class_anomaly_counts[c] / max(1, class_box_counts[c])) * 100 for c in all_known_classes]
    bars2 = axs[0, 1].bar(all_known_classes, cls_rates, color=["#3498db", "#9b59b6", "#e67e22"][:len(all_known_classes)], edgecolor="black", alpha=0.85)
    axs[0, 1].set_title("Anomaly Rate (%) by Object Class", fontsize=11, fontweight="bold")
    axs[0, 1].set_ylabel("Anomaly Percentage (%)")
    axs[0, 1].set_ylim(0, max(cls_rates + [10]) * 1.25)
    for bar in bars2:
        axs[0, 1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, f"{bar.get_height():.1f}%", ha="center", fontsize=9, fontweight="bold")
else:
    axs[0, 1].text(0.5, 0.5, "No Classes Available", ha="center", va="center")

# Panel 3: Aspect Ratio (w/h) Distribution Across Classes
for c in all_known_classes:
    c_boxes = [a for anns in image_source.values() for a in anns if str(a.get("category_name", "")).lower() == c]
    if c_boxes:
        c_ratios = [a["bbox"][2] / max(a["bbox"][3], 0.1) for a in c_boxes if len(a.get("bbox", [])) == 4 and a["bbox"][2] > 0 and a["bbox"][3] > 0]
        c_ratios = [r for r in c_ratios if r <= 15.0]
        if c_ratios:
            axs[1, 0].hist(c_ratios, bins=40, alpha=0.55, label=f"{c} (n={len(c_ratios):,})", edgecolor="none")
axs[1, 0].set_title("Aspect Ratio (w/h) Distribution by Class", fontsize=11, fontweight="bold")
axs[1, 0].set_xlabel("Aspect Ratio (w/h)")
axs[1, 0].set_ylabel("Count")
axs[1, 0].legend(loc="upper right")

# Panel 4: Blur Sharpness Score Distribution
blur_scores_all = list(_image_blur_scores.values())
if blur_scores_all:
    axs[1, 1].hist([s for s in blur_scores_all if s < 1000], bins=50, color="#2ecc71", edgecolor="black", alpha=0.75)
    axs[1, 1].axvline(BLUR_THRESHOLD, color="red", linestyle="--", linewidth=2, label=f"Blur Cutoff ({BLUR_THRESHOLD})")
    axs[1, 1].set_title(f"Image Blur Sharpness Distribution (Laplacian Var, N={len(blur_scores_all):,})", fontsize=11, fontweight="bold")
    axs[1, 1].set_xlabel("Sharpness Score (Lower = More Blurry)")
    axs[1, 1].set_ylabel("Number of Images")
    axs[1, 1].legend(loc="upper right")
else:
    axs[1, 1].text(0.5, 0.5, "No Blur Data", ha="center", va="center")

plt.tight_layout()
diagnostic_plot_path = OUTPUT_ANOMALIES_DIR / "anomaly_diagnostics_report.png"
plt.savefig(diagnostic_plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"✓ Saved diagnostic plots to unified version folder: {diagnostic_plot_path}")


In [ ]:
# CELL 7 - Visual Display of Detected Anomaly Samples Across ALL Classes
import os, sys, json
from PIL import Image, ImageDraw, ImageFont
from IPython.display import display
from collections import defaultdict, Counter
from pathlib import Path
import urllib.parse

def normalize_anomaly_category(reason: str) -> str:
    """Canonical name for anomaly rule grouping."""
    r = str(reason).split("(")[0].strip()
    if "Duplicate box" in r:
        return "Duplicate Box Overlap"
    if "Micro-click" in r:
        return "Micro-Click (<2px)"
    if "Degenerate box" in r:
        return "Degenerate Box (Zero Dim)"
    if "Cropped box at image border" in r or "border" in r.lower():
        return "Edge-Cropped / Border-Truncated Box"
    if "Blurry" in r:
        return "Severe Image Blur"
    if "Giant box" in r:
        return "Giant Box (>95% Image)"
    if "Dense cluster" in r:
        return "Dense Cluster Overlap"
    if "Loose shelf padding" in r:
        return "Loose Shelf Padding"
    if "Extreme Vertical" in r:
        return r
    if "Extreme Horizontal Sliver" in r:
        return r
    if "Out-of-Bounds" in r:
        return r
    if "Unusually Giant Area" in r:
        return r
    return r

def display_anomaly_breakdown_and_samples(
    flagged_anomalies: list,
    images_dir: Path,
    samples_per_anomaly: int = 5,
    max_display_width: int = 750
):
    """
    Renders visual sample images for EVERY detected anomaly category across ALL classes.
    Draws highlighted bounding boxes and metadata overlays so the user can visually inspect
    each anomaly pattern before deciding whether to exclude it from training.
    """
    if not flagged_anomalies:
        print("[INFO] No anomalies flagged in dataset. All annotations are clean.")
        return

    # Group anomalies by normalized category / rule
    anomalies_by_category = defaultdict(list)
    for anom in flagged_anomalies:
        reasons = anom.get("reasons", ["Unspecified anomaly"])
        primary = anom.get("primary_rule") or (reasons[0] if reasons else "Unknown")
        norm_cat = normalize_anomaly_category(primary)
        anomalies_by_category[norm_cat].append(anom)

    sorted_categories = sorted(
        anomalies_by_category.keys(),
        key=lambda c: len(anomalies_by_category[c]),
        reverse=True
    )

    print("=" * 85)
    print("VISUAL ANOMALY INSPECTION SAMPLES (ALL CLASSES & CATEGORIES)")
    print(f"Detected {len(sorted_categories)} anomaly categories across {len(flagged_anomalies):,} flagged boxes.")
    print(f"Rendering up to {samples_per_anomaly} sample images per category with bounding box overlays.")
    print("=" * 85)

    for cat_idx, rule in enumerate(sorted_categories, 1):
        candidates = anomalies_by_category[rule]
        total_for_rule = len(candidates)

        print()
        print(f"[{cat_idx}/{len(sorted_categories)}] ANOMALY CATEGORY: '{rule}' ({total_for_rule:,} boxes flagged)")
        print("-" * 85)

        # Group candidates by unique image
        imgs_for_rule = defaultdict(list)
        for anom in candidates:
            img_url = anom.get("image_url") or anom.get("annotation", {}).get("file_name", "unknown")
            imgs_for_rule[img_url].append(anom)

        rendered_count = 0
        for img_url, matching_anoms in imgs_for_rule.items():
            if rendered_count >= samples_per_anomaly:
                break

            local_path = resolve_image_path(img_url)
            if not local_path or not os.path.exists(local_path):
                continue

            try:
                pil_img = Image.open(local_path).convert("RGB")
            except Exception as e:
                continue

            draw = ImageDraw.Draw(pil_img)
            im_w, im_h = pil_img.size
            fname = Path(local_path).name

            # 1. First draw clean / non-anomalous annotations in GREEN / CYAN
            clean_in_img = [
                a for a in clean_annotations
                if (a.get("image_url") == img_url or a.get("file_name") == fname)
            ]
            for ca in clean_in_img:
                cb = ca.get("bbox", [])
                if len(cb) == 4 and cb[2] > 0 and cb[3] > 0:
                    cx, cy, cw, ch = float(cb[0]), float(cb[1]), float(cb[2]), float(cb[3])
                    draw.rectangle([cx, cy, cx + cw, cy + ch], outline="#00e676", width=2)
                    c_tag = str(ca.get("category_name", "clean"))
                    draw.text((cx + 2, max(0, cy - 12)), f"clean: {c_tag}", fill="#00e676")

            # 2. Draw highlighted ANOMALY annotations in BOLD RED with label badge
            matching_tags = []
            for anom in matching_anoms:
                b = anom.get("bbox", [])
                tag_name = anom.get("category_name", "Unknown")
                matching_tags.append(tag_name)
                
                if len(b) == 4:
                    bx, by, bw, bh = float(b[0]), float(b[1]), float(b[2]), float(b[3])
                    if bw > 0 and bh > 0:
                        # Draw thick warning box in RED
                        draw.rectangle([bx, by, bx + bw, by + bh], outline="#ff1744", width=4)
                        
                        # Label text with rule and dimensions
                        label_text = f"ANOMALY [{tag_name}]: {rule} (w={bw:.0f}, h={bh:.0f})"
                        text_y = max(0, by - 16)
                        draw.rectangle([bx, text_y, bx + min(im_w - bx, len(label_text)*7 + 10), text_y + 16], fill="#d50000")
                        draw.text((bx + 3, text_y + 2), label_text, fill="#ffffff")

            rendered_count += 1
            print(f"  • Sample #{rendered_count} | Image: {fname} ({im_w}x{im_h})")
            print(f"    - Target Class: {', '.join(set(matching_tags))} ({len(matching_anoms)} anomaly boxes)")
            print(f"    - First Reason: {matching_anoms[0].get('reasons', [''])[0]}")

            # Scale for notebook inline display
            if im_w > max_display_width:
                scale = max_display_width / im_w
                display_img = pil_img.resize((max_display_width, int(im_h * scale)), Image.BILINEAR)
            else:
                display_img = pil_img

            display(display_img)

        if rendered_count == 0:
            print(f"  [INFO] No local cached image files available for rule '{rule}'.")

    print()
    print("=" * 85)
    print("✓ Anomaly sample inspection complete across all classes & categories.")
    print("=" * 85)

# Execute the sample inspector
samples_to_show = globals().get("SHOW_ANOMALY_SAMPLES_PER_CATEGORY", 5)
display_anomaly_breakdown_and_samples(flagged_anomalies, IMAGES_DIR, samples_per_anomaly=samples_to_show)


In [ ]:
# CELL 8 - Export Manifest & Sanitized Dataset
manifest_path = OUTPUT_ANOMALIES_DIR / "anomalies_manifest.json"
with open(manifest_path, "w", encoding="utf-8") as f:
    json.dump({
        "flagged_count": len(flagged_anomalies),
        "total_audited": total_boxes,
        "clean_count": len(clean_annotations),
        "anomalies": flagged_anomalies
    }, f, indent=2)

logger.info(f"✓ Saved anomalies manifest: {manifest_path}")

clean_coco = {
    "images": list({a.get("image_id"): {
        "id": a.get("image_id"),
        "file_name": a.get("file_name", f"{a.get('image_id')}.jpg"),
        "width": int(a.get("img_w", 1920)),
        "height": int(a.get("img_h", 1080)),
        "coco_url": a.get("image_url")
    } for a in clean_annotations}.values()),
    "annotations": clean_annotations,
    "categories": categories_list
}
clean_json_path = CLEANED_DATASET_DIR / "_annotations.coco.json"
with open(clean_json_path, "w", encoding="utf-8") as f:
    json.dump(clean_coco, f, indent=2)

logger.info(f"✓ Saved sanitized dataset: {clean_json_path} ({len(clean_coco['annotations'])} clean annotations)")


In [ ]:
# CELL 9 - Step B Verification on Cleaned Dataset
def load_coco_info(annotation_path: str) -> dict:
    """Parse a COCO annotation file and return summary statistics."""
    with open(annotation_path) as f:
        data = json.load(f)
    categories = {c["id"]: c["name"] for c in data.get("categories", [])}
    num_images = len(data.get("images", []))
    num_anns = len(data.get("annotations", []))
    ann_per_cat: dict = {}
    for ann in data.get("annotations", []):
        cat_name = categories.get(ann["category_id"], "unknown")
        ann_per_cat[cat_name] = ann_per_cat.get(cat_name, 0) + 1
    return {
        "num_images": num_images,
        "num_anns": num_anns,
        "num_classes": len(categories),
        "categories": categories,
        "ann_per_cat": ann_per_cat,
        "raw": data,
    }

# Verify sanitized output file exists
target_cleaned_ann = clean_json_path if 'clean_json_path' in globals() else CLEANED_DATASET_DIR / "_annotations.coco.json"
if target_cleaned_ann.exists():
    cleaned_info = load_coco_info(str(target_cleaned_ann))
    logger.info("=" * 65)
    logger.info(f"Cleaned Dataset Audit: {cleaned_info['num_images']} images, {cleaned_info['num_anns']} annotations, {cleaned_info['num_classes']} classes")
    logger.info(f"Classes: {cleaned_info['categories']}")
    logger.info("=" * 65)
else:
    logger.info(f"Cleaned dataset file {target_cleaned_ann} not generated yet.")


In [ ]:
import json, os, sys
from pathlib import Path
from collections import Counter, defaultdict
from typing import Tuple

# CELL 10 - Anomaly Selection & Pipeline Export Configuration (Gate to Step 03)
# ==============================================================================
# PIPELINE CONTROL: USER DECIDES WHAT CAN BE TREATED AS ANOMALIES (ALL CLASSES)
# By default, ONLY severe blur images are True (excluded from training).
# All other anomaly categories default to False (kept in training).
# You can toggle any rule to True or False below based on the audit & samples above.
# ==============================================================================

EXCLUDE_ANOMALIES_FROM_TRAINING = True

CONSIDER_ANOMALIES = {
    # --------------------------------------------------------------------------
    # 1. BLUR ANOMALY (DEFAULT: TRUE -> Exclude severe blur images from training)
    # --------------------------------------------------------------------------
    'Severe Image Blur': True,
    'Blurry image': True,

    # --------------------------------------------------------------------------
    # 2. HARD BOX ERRORS (DEFAULT: FALSE -> User decides whether to exclude)
    # --------------------------------------------------------------------------
    'Degenerate Box (Zero Dim)': False,
    'Degenerate box': False,
    'Malformed bbox dimensions': False,
    'Out-of-Bounds Coordinates': False,
    '[location_tag] Out-of-Bounds Coordinates': False,
    '[blue_bay] Out-of-Bounds Coordinates': False,
    '[blue_aisle] Out-of-Bounds Coordinates': False,

    # --------------------------------------------------------------------------
    # 3. MULTI-CLASS FORMAT & GEOMETRY PATTERNS (DEFAULT: FALSE -> Keep in training)
    # --------------------------------------------------------------------------
    'Extreme Vertical Aspect Ratio': False,
    '[location_tag] Extreme Vertical Aspect Ratio': False,
    '[blue_bay] Extreme Vertical Aspect Ratio': False,
    '[blue_aisle] Extreme Vertical Aspect Ratio': False,

    'Extreme Horizontal Sliver': False,
    '[location_tag] Extreme Horizontal Sliver': False,
    '[blue_bay] Extreme Horizontal Sliver': False,
    '[blue_aisle] Extreme Horizontal Sliver': False,

    'Unusually Giant Area': False,
    '[location_tag] Unusually Giant Area': False,
    '[blue_bay] Unusually Giant Area': False,
    '[blue_aisle] Unusually Giant Area': False,

    # --------------------------------------------------------------------------
    # 4. DOMAIN EDGE CASES & CLUSTERS (DEFAULT: FALSE -> Keep in training)
    # --------------------------------------------------------------------------
    'Edge-Cropped / Border-Truncated Box': False,
    'Cropped box at image border': False,
    'Micro-Click (<2px)': False,
    'Micro-click': False,
    'Dense Cluster Overlap': False,
    'Dense cluster overlap': False,
    'Duplicate Box Overlap': False,
    'Duplicate box with IoU': False,
    'Loose Shelf Padding': False,
    'Loose shelf padding across multiple bays': False,
    'Giant Box (>95% Image)': False,
    'Giant box covering >95% image': False,
}

print('=' * 85)
print('[CELL 10] ANOMALY SELECTION & PIPELINE EXPORT CONFIGURATION')
print('=' * 85)
print(f'Mode: {"EXCLUDE ANOMALIES FROM TRAINING" if EXCLUDE_ANOMALIES_FROM_TRAINING else "AUDIT ONLY (Retain all annotations)"}')
print('Default Policy: ONLY Severe Image Blur is treated as anomaly (True); all others are kept (False).')
print()
print('Active Anomaly Filters (User Configuration):')
for rule_name, active in CONSIDER_ANOMALIES.items():
    status_str = 'EXCLUDE from training' if active else 'KEEP in training'
    mark = 'X' if active else ' '
    print(f'  [{mark}] {rule_name:<50} -> {status_str}')
print('-' * 85)

# Filter anomalies according to CONSIDER_ANOMALIES configuration
final_excluded_anomalies = []
final_retained_annotations = []
excluded_counts_by_rule = Counter()
excluded_counts_by_class = Counter()

def should_exclude_anomaly(anom: dict) -> Tuple[bool, str]:
    """Evaluates whether an anomaly should be excluded based on CONSIDER_ANOMALIES.
    Only categories where CONSIDER_ANOMALIES evaluates to True will be excluded.
    """
    if not EXCLUDE_ANOMALIES_FROM_TRAINING:
        return False, ""
    cat_name = str(anom.get("category_name", "")).lower().strip()
    for r in anom.get("reasons", []):
        r_prefix = r.split("(")[0].strip()
        norm_r = normalize_anomaly_category(r_prefix)
        
        # 1. Exact match with reason prefix (e.g. '[location_tag] Extreme Vertical Aspect Ratio')
        if r_prefix in CONSIDER_ANOMALIES:
            if CONSIDER_ANOMALIES[r_prefix]:
                return True, r_prefix
            else:
                continue
                
        # 2. Match with normalized category name (e.g. 'Severe Image Blur')
        if norm_r in CONSIDER_ANOMALIES:
            if CONSIDER_ANOMALIES[norm_r]:
                return True, norm_r
            else:
                continue
                
        # 3. Class-prefixed generic rule match (e.g. '[location_tag] Out-of-Bounds Coordinates')
        if "]" in r_prefix:
            generic_suffix = r_prefix.split("]")[-1].strip()
            if generic_suffix in CONSIDER_ANOMALIES:
                if CONSIDER_ANOMALIES[generic_suffix]:
                    return True, generic_suffix
                else:
                    continue
                    
        # 4. Fallback keyword checks against CONSIDER_ANOMALIES
        if "Blurry" in r_prefix:
            if CONSIDER_ANOMALIES.get("Severe Image Blur", False) or CONSIDER_ANOMALIES.get("Blurry image", False):
                return True, "Severe Image Blur"
        elif "Out-of-Bounds" in r_prefix:
            if CONSIDER_ANOMALIES.get("Out-of-Bounds Coordinates", False):
                return True, "Out-of-Bounds Coordinates"
        elif "Degenerate" in r_prefix:
            if CONSIDER_ANOMALIES.get("Degenerate Box (Zero Dim)", False) or CONSIDER_ANOMALIES.get("Degenerate box", False):
                return True, "Degenerate Box (Zero Dim)"
        elif "Malformed" in r_prefix:
            if CONSIDER_ANOMALIES.get("Malformed bbox dimensions", False):
                return True, "Malformed bbox dimensions"
                
    return False, ""

for anom in flagged_anomalies:
    exclude, matched_rule = should_exclude_anomaly(anom)
    c_name = str(anom.get("category_name", "unknown")).lower()
    if exclude:
        final_excluded_anomalies.append(anom)
        excluded_counts_by_rule[matched_rule] += 1
        excluded_counts_by_class[c_name] += 1
    else:
        final_retained_annotations.append(anom.get("annotation", anom))

# Add base clean annotations
final_retained_annotations.extend(clean_annotations)

retained_pct = (len(final_retained_annotations) / max(1, total_boxes)) * 100
excluded_pct = (len(final_excluded_anomalies) / max(1, total_boxes)) * 100
print(f"Final Training Box Count: {len(final_retained_annotations):,} / {total_boxes:,} ({retained_pct:.1f}% retained)")
print(f"Excluded Anomaly Boxes:   {len(final_excluded_anomalies):,} ({excluded_pct:.1f}% excluded)")

if excluded_counts_by_rule:
    print()
    print("Breakdown of Excluded Boxes by Rule:")
    for r_name, r_cnt in excluded_counts_by_rule.most_common():
        print(f"  - {r_name:<48}: {r_cnt:,} boxes")

if excluded_counts_by_class:
    print()
    print("Breakdown of Excluded Boxes by Class:")
    for c_name, c_cnt in excluded_counts_by_class.most_common():
        print(f"  - {c_name:<20}: {c_cnt:,} boxes excluded")
print("-" * 85)

# Export the active training dataset into VERSION_DIR/cleaned_dataset/
final_training_coco = {
    'info': {
        'description': f'RF-DETR Multi-Class Training Dataset ({"Cleaned" if EXCLUDE_ANOMALIES_FROM_TRAINING else "Full Audit"})',
        'version': VERSION_TAG,
        'exclude_anomalies': EXCLUDE_ANOMALIES_FROM_TRAINING,
        'active_rules': CONSIDER_ANOMALIES
    },
    'images': list({
        a.get('image_id', idx): {
            'id': a.get('image_id', idx),
            'file_name': extract_clean_filename(a.get('file_name') or (raw_images.get(a.get('image_id'), {}).get('file_name') if 'raw_images' in globals() else None) or a.get('image_url'), default_id=a.get('image_id', idx)) if 'extract_clean_filename' in globals() else (a.get('file_name') or f"{a.get('image_id', idx)}.jpg"),
            'width': int(a.get('img_w') or (raw_images.get(a.get('image_id'), {}).get('width', 1920))),
            'height': int(a.get('img_h') or (raw_images.get(a.get('image_id'), {}).get('height', 1080))),
            'coco_url': a.get('image_url') or a.get('file_name')
        } for idx, a in enumerate(final_retained_annotations)
    }.values()),
    'annotations': [
        {
            'id': a.get('id', idx + 1),
            'image_id': a.get('image_id', 0),
            'category_id': a.get('category_id', 0),
            'category_name': a.get('category_name', 'location_tag'),
            'bbox': a.get('bbox', [0, 0, 0, 0]),
            'area': a.get('area', 0.0),
            'iscrowd': 0,
            'segmentation': a.get('segmentation', [])
        }
        for idx, a in enumerate(final_retained_annotations)
    ],
    'categories': categories_list
}

training_export_path = CLEANED_DATASET_DIR / '_annotations.coco.json'
with open(training_export_path, 'w', encoding='utf-8') as f:
    json.dump(final_training_coco, f, indent=2)

# Also mirror to REPO_ROOT / cleaned_dataset so Step 03 finds it regardless of current working directory
mirror_export_path = REPO_ROOT / 'cleaned_dataset' / '_annotations.coco.json'
mirror_export_path.parent.mkdir(parents=True, exist_ok=True)
with open(mirror_export_path, 'w', encoding='utf-8') as f:
    json.dump(final_training_coco, f, indent=2)

# Save updated manifest into VERSION_DIR/anomalies_audit/
manifest_export_path = OUTPUT_ANOMALIES_DIR / 'anomalies_manifest.json'
with open(manifest_export_path, 'w', encoding='utf-8') as f:
    json.dump({
        'version': VERSION_TAG,
        'exclude_anomalies_from_training': EXCLUDE_ANOMALIES_FROM_TRAINING,
        'consider_anomalies_rules': CONSIDER_ANOMALIES,
        'total_audited': total_boxes,
        'clean_retained_count': len(final_retained_annotations),
        'excluded_anomalies_count': len(final_excluded_anomalies),
        'excluded_counts_by_rule': dict(excluded_counts_by_rule),
        'excluded_counts_by_class': dict(excluded_counts_by_class),
        'excluded_anomalies': final_excluded_anomalies
    }, f, indent=2)

print()
print(f"✓ Saved training dataset to unified version folder: {training_export_path}")
print(f"✓ Saved updated manifest to unified version folder: {manifest_export_path}")
print('=' * 85)
print('PROCEED TO STEP 03: Run 03_stratified_train_val_test_split.ipynb to generate stratified splits.')
print('=' * 85)
